# Generative Adversarial Network (GAN) on MNIST

This notebook implements the original GAN paper (Goodfellow et al., 2014) from scratch using PyTorch.

**Goal:** Train a Generator to produce handwritten digit images that are indistinguishable from real MNIST digits.

**Architecture recap:**
- **Generator G**: takes random noise z ~ N(0,1) → outputs a 28×28 fake image
- **Discriminator D**: takes a 28×28 image → outputs a scalar probability (real vs fake)
- They play a minimax game:

$$\min_G \max_D \; V(D,G) = \mathbb{E}_{x}[\log D(x)] + \mathbb{E}_{z}[\log(1 - D(G(z)))]$$

## Step 1 — Imports and Configuration

We use:
- `torch` + `torch.nn` for building and training the networks
- `torchvision` for the MNIST dataset and image utilities
- `matplotlib` for visualizing generated images during training

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# ── Hyperparameters ──────────────────────────────────────────────────────────
LATENT_DIM   = 100      # dimension of the noise vector z
HIDDEN_DIM   = 256      # hidden layer size in both G and D
IMG_DIM      = 28 * 28  # MNIST images are 28×28 = 784 pixels (flattened)
BATCH_SIZE   = 128
NUM_EPOCHS   = 50
LR           = 2e-4     # learning rate for both G and D
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Using device: {DEVICE}')
print(f'Latent dim: {LATENT_DIM}, Image dim: {IMG_DIM}')

## Step 2 — Load the MNIST Dataset

We normalize pixel values from [0, 255] to [-1, 1].

**Why [-1, 1]?** The Generator's output layer uses `Tanh`, which outputs values in [-1, 1]. Matching the data range to the activation range makes training more stable.

In [ ]:
# Transform: convert to tensor and normalize to [-1, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,))  # (x - 0.5) / 0.5 maps [0,1] → [-1,1]
])

train_dataset = datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,      # shuffle so each batch is a random mini-batch
    num_workers=2,
    pin_memory=True    # speeds up CPU→GPU data transfer
)

print(f'Dataset size : {len(train_dataset):,} images')
print(f'Batches/epoch: {len(train_loader)}')

# Visualize a batch of real images
real_batch, _ = next(iter(train_loader))
grid = make_grid(real_batch[:32], nrow=8, normalize=True)
plt.figure(figsize=(10, 4))
plt.imshow(grid.permute(1, 2, 0), cmap='gray')
plt.title('Real MNIST images (normalized to [-1, 1])')
plt.axis('off')
plt.tight_layout()
plt.show()

## Step 3 — Build the Generator G

The Generator maps a latent noise vector **z ∈ ℝ¹⁰⁰** to a fake image **G(z) ∈ ℝ⁷⁸⁴**.

Architecture:
```
z (100) → Linear → BN → LeakyReLU
        → Linear → BN → LeakyReLU
        → Linear → BN → LeakyReLU
        → Linear → Tanh → image (784)
```

**Key design choices:**
- **Batch Normalization**: stabilizes training by normalizing layer inputs
- **LeakyReLU**: avoids dead neurons (allows small gradient for negative values)
- **Tanh output**: maps outputs to [-1, 1], matching normalized MNIST pixel range

In [ ]:
class Generator(nn.Module):
    """
    Generator G(z; θ_g): maps noise z → fake image.
    
    In the original GAN paper, G is a multilayer perceptron (MLP).
    We follow that exactly: fully connected layers with batch norm.
    """
    def __init__(self, latent_dim: int, hidden_dim: int, img_dim: int):
        super().__init__()
        
        def block(in_dim, out_dim, batch_norm=True):
            """A single generator block: Linear → [BatchNorm] → LeakyReLU"""
            layers = [nn.Linear(in_dim, out_dim)]
            if batch_norm:
                layers.append(nn.BatchNorm1d(out_dim, momentum=0.8))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        
        self.model = nn.Sequential(
            # z (100) → 256
            *block(latent_dim,        hidden_dim,     batch_norm=False),
            # 256 → 512
            *block(hidden_dim,        hidden_dim * 2),
            # 512 → 1024
            *block(hidden_dim * 2,    hidden_dim * 4),
            # 1024 → 784 (image), no batch norm on output layer
            nn.Linear(hidden_dim * 4, img_dim),
            nn.Tanh()   # output ∈ [-1, 1]
        )
    
    def forward(self, z):
        """
        z: (batch_size, latent_dim)
        returns: (batch_size, img_dim)  — flattened fake images
        """
        return self.model(z)


G = Generator(LATENT_DIM, HIDDEN_DIM, IMG_DIM).to(DEVICE)
print('Generator architecture:')
print(G)
total_params = sum(p.numel() for p in G.parameters())
print(f'\nTotal trainable parameters: {total_params:,}')

## Step 4 — Build the Discriminator D

The Discriminator maps an image **x ∈ ℝ⁷⁸⁴** to a scalar **D(x) ∈ [0, 1]**.

Architecture:
```
x (784) → Linear → LeakyReLU → Dropout
        → Linear → LeakyReLU → Dropout
        → Linear → LeakyReLU → Dropout
        → Linear → Sigmoid → scalar
```

**Key design choices:**
- **No Batch Normalization** in D: as mentioned in the paper, this can hurt D's ability to distinguish single-image statistics
- **Dropout**: regularization — prevents D from memorizing training data
- **LeakyReLU (slope=0.2)**: the paper explicitly uses this to improve gradient flow
- **Sigmoid output**: maps to [0, 1] so it can be interpreted as a probability

In [ ]:
class Discriminator(nn.Module):
    """
    Discriminator D(x; θ_d): maps image x → P(x is real).
    
    Outputs a scalar in [0, 1]:
      D(x) ≈ 1  →  D thinks x is REAL
      D(x) ≈ 0  →  D thinks x is FAKE
    """
    def __init__(self, img_dim: int, hidden_dim: int):
        super().__init__()
        
        def block(in_dim, out_dim, dropout=0.3):
            """A single discriminator block: Linear → LeakyReLU → Dropout"""
            return [
                nn.Linear(in_dim, out_dim),
                nn.LeakyReLU(0.2, inplace=True),
                nn.Dropout(dropout)
            ]
        
        self.model = nn.Sequential(
            # 784 → 512
            *block(img_dim,           hidden_dim * 2),
            # 512 → 256
            *block(hidden_dim * 2,    hidden_dim),
            # 256 → 128
            *block(hidden_dim,        hidden_dim // 2),
            # 128 → 1
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()   # output ∈ [0, 1] = probability of being real
        )
    
    def forward(self, x):
        """
        x: (batch_size, img_dim)
        returns: (batch_size, 1)  — probability each image is real
        """
        return self.model(x)


D = Discriminator(IMG_DIM, HIDDEN_DIM).to(DEVICE)
print('Discriminator architecture:')
print(D)
total_params = sum(p.numel() for p in D.parameters())
print(f'\nTotal trainable parameters: {total_params:,}')

## Step 5 — Loss Function and Optimizers

Both G and D use **Binary Cross-Entropy (BCE)** loss, which is exactly log(D(x)) and log(1-D(G(z))).

$$\text{BCELoss}(\hat{y}, y) = -\left[ y \log(\hat{y}) + (1-y) \log(1 - \hat{y}) \right]$$

**D's objective** (maximize V — so we minimize the negative):
- Real images: label = 1 → minimize `-log(D(x_real))`
- Fake images: label = 0 → minimize `-log(1 - D(G(z)))`

**G's objective** (practical version: maximize log D(G(z))):
- Fake images: label = 1 (trick D) → minimize `-log(D(G(z)))`

This is the **non-saturating trick** from the paper.

In [ ]:
# Binary Cross-Entropy: this IS the GAN loss function
criterion = nn.BCELoss()

# Separate optimizers for G and D (they take turns updating)
optimizer_G = optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
# Note: beta1=0.5 is common for GANs (default 0.9 can cause instability)

# Fixed noise to track G's progress over epochs (always same z → reproducible)
FIXED_NOISE = torch.randn(64, LATENT_DIM, device=DEVICE)

# Pre-allocate label tensors (reuse each batch to avoid re-allocation)
REAL_LABEL = 1.0
FAKE_LABEL = 0.0

print('Loss function : BCELoss (= GAN objective)')
print('Optimizer G   : Adam(lr=2e-4, β₁=0.5)')
print('Optimizer D   : Adam(lr=2e-4, β₁=0.5)')
print(f'Fixed noise   : {FIXED_NOISE.shape}  (used to track G quality)')

## Step 6 — Weight Initialization

GANs are sensitive to initialization. We initialize all weights from N(0, 0.02) as is standard practice.

Poor initialization can cause:
- Mode collapse (G always outputs the same image)
- Training divergence (D wins immediately, no gradient for G)

In [ ]:
def weights_init(m):
    """
    Initialize weights of Linear and BatchNorm layers.
    Based on the DCGAN paper convention, widely adopted.
    """
    classname = m.__class__.__name__
    if classname == 'Linear':
        nn.init.normal_(m.weight.data, 0.0, 0.02)  # small normal distribution
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0)
    elif classname == 'BatchNorm1d':
        nn.init.normal_(m.weight.data, 1.0, 0.02)  # BN scale near 1
        nn.init.constant_(m.bias.data, 0)           # BN bias = 0

G.apply(weights_init)
D.apply(weights_init)
print('Weights initialized from N(0, 0.02)')

## Step 7 — Helper Functions

Before the training loop, we define:
- `show_generated()`: visualize G's output from the fixed noise vector
- `plot_losses()`: plot D and G loss curves side by side

In [ ]:
@torch.no_grad()  # no gradient tracking during visualization
def show_generated(epoch, fixed_noise):
    """Show 64 images generated from the fixed noise vector."""
    G.eval()
    fake_images = G(fixed_noise)                          # (64, 784)
    fake_images = fake_images.view(-1, 1, 28, 28)         # reshape to (64, 1, 28, 28)
    grid = make_grid(fake_images, nrow=8, normalize=True, value_range=(-1, 1))
    
    plt.figure(figsize=(10, 10))
    plt.imshow(grid.cpu().permute(1, 2, 0), cmap='gray')
    plt.title(f'Epoch {epoch} — Generated images', fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    G.train()


def plot_losses(g_losses, d_losses):
    """Plot generator and discriminator losses over training."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].plot(g_losses, label='G loss', color='purple', alpha=0.8, linewidth=0.8)
    axes[0].plot(d_losses, label='D loss', color='coral',  alpha=0.8, linewidth=0.8)
    axes[0].set_title('Training losses (per batch)')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Smoothed (per-epoch average)
    batches = len(g_losses) // len(set(range(len(g_losses))))
    n = max(len(train_loader), 1)
    g_ep = [np.mean(g_losses[i*n:(i+1)*n]) for i in range(len(g_losses)//n)]
    d_ep = [np.mean(d_losses[i*n:(i+1)*n]) for i in range(len(d_losses)//n)]
    
    axes[1].plot(g_ep, label='G loss (epoch avg)', color='purple', linewidth=2)
    axes[1].plot(d_ep, label='D loss (epoch avg)', color='coral',  linewidth=2)
    axes[1].set_title('Training losses (per epoch average)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print('Helper functions defined.')

## Step 8 — The Training Loop

This is the core of the GAN. For each batch we do **two updates**:

### Update D (maximize V)
1. Feed **real images** → D, labels = 1 → compute `loss_D_real = BCE(D(x), 1)`
2. Feed **fake images** G(z) → D, labels = 0 → compute `loss_D_fake = BCE(D(G(z)), 0)`
3. Total D loss = `loss_D_real + loss_D_fake` → backprop → update θ_d

### Update G (maximize log D(G(z)) — non-saturating trick)
1. Feed fresh fake images G(z) → D, labels = **1** (we want D to say real)
2. `loss_G = BCE(D(G(z)), 1)` → backprop **through D** → update θ_g

> **Important:** when updating G, we do **not** update D. We call `optimizer_D.zero_grad()` before D's step and `optimizer_G.zero_grad()` before G's step.

In [ ]:
g_losses = []   # G loss per batch (for plotting)
d_losses = []   # D loss per batch (for plotting)

print(f'Starting training for {NUM_EPOCHS} epochs on {DEVICE}...')
print('=' * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_d_loss = 0.0
    epoch_g_loss = 0.0
    
    for batch_idx, (real_imgs, _) in enumerate(train_loader):
        
        batch_size = real_imgs.size(0)
        
        # Flatten images: (B, 1, 28, 28) → (B, 784)
        real_imgs = real_imgs.view(batch_size, -1).to(DEVICE)
        
        # ── Labels ──────────────────────────────────────────────────────────
        real_labels = torch.full((batch_size, 1), REAL_LABEL, device=DEVICE)
        fake_labels = torch.full((batch_size, 1), FAKE_LABEL, device=DEVICE)
        
        # ════════════════════════════════════════════════════════════════════
        # PHASE 1: Update D — maximize log D(x) + log(1 - D(G(z)))
        # ════════════════════════════════════════════════════════════════════
        optimizer_D.zero_grad()
        
        # 1a. Real images → D should output 1
        d_real_output = D(real_imgs)                       # (B, 1)
        loss_D_real   = criterion(d_real_output, real_labels)  # -log(D(x))
        
        # 1b. Fake images → D should output 0
        noise       = torch.randn(batch_size, LATENT_DIM, device=DEVICE)  # z ~ N(0,I)
        fake_imgs   = G(noise).detach()   # .detach() stops gradients flowing into G here
        d_fake_output = D(fake_imgs)                       # (B, 1)
        loss_D_fake   = criterion(d_fake_output, fake_labels)  # -log(1 - D(G(z)))
        
        # 1c. Total D loss = sum of both terms
        loss_D = loss_D_real + loss_D_fake                 # = -V(D,G) (we minimize)
        loss_D.backward()
        optimizer_D.step()
        
        # ════════════════════════════════════════════════════════════════════
        # PHASE 2: Update G — maximize log D(G(z))  [non-saturating trick]
        # ════════════════════════════════════════════════════════════════════
        optimizer_G.zero_grad()
        
        # Generate new fake images (fresh z each time)
        noise     = torch.randn(batch_size, LATENT_DIM, device=DEVICE)
        fake_imgs = G(noise)                               # (B, 784)
        
        # G wants D to classify fake images as REAL → label = 1
        d_output_for_g = D(fake_imgs)                     # (B, 1)
        loss_G = criterion(d_output_for_g, real_labels)   # -log(D(G(z)))
        
        loss_G.backward()   # gradients flow: loss → D → G
        optimizer_G.step()
        
        # ── Record losses ────────────────────────────────────────────────────
        g_losses.append(loss_G.item())
        d_losses.append(loss_D.item())
        epoch_d_loss += loss_D.item()
        epoch_g_loss += loss_G.item()
    
    # ── End of epoch ────────────────────────────────────────────────────────
    avg_d = epoch_d_loss / len(train_loader)
    avg_g = epoch_g_loss / len(train_loader)
    
    # D(x): average discriminator score on real images (want ≈ 0.5 at equilibrium)
    with torch.no_grad():
        d_x  = D(real_imgs).mean().item()
        d_gz = D(G(torch.randn(batch_size, LATENT_DIM, device=DEVICE))).mean().item()
    
    print(f'Epoch [{epoch:3d}/{NUM_EPOCHS}]  '
          f'D loss: {avg_d:.4f}  '
          f'G loss: {avg_g:.4f}  '
          f'D(x): {d_x:.3f}  '
          f'D(G(z)): {d_gz:.3f}')
    
    # Visualize generated images every 10 epochs
    if epoch % 10 == 0 or epoch == 1:
        show_generated(epoch, FIXED_NOISE)

print('\nTraining complete!')

## Step 9 — Visualize Training Losses

**What to look for:**
- D loss should stay around **~0.5–1.4** (log(2) ≈ 0.693 at equilibrium)
- G loss should decrease over time as it learns to fool D
- If D loss → 0 quickly: D is dominating — G gets no gradient signal (bad)
- If G loss → 0 quickly: G is dominating — mode collapse risk (bad)

Healthy training has both losses oscillating around equilibrium.

In [ ]:
plot_losses(g_losses, d_losses)

## Step 10 — Explore the Latent Space

A key property of GANs: the latent space **z** is continuous and smooth.

We can interpolate between two noise vectors z₁ and z₂ and see the Generator smoothly morph between two generated images. This shows G has learned a meaningful representation — it's not just memorizing images.

In [ ]:
@torch.no_grad()
def interpolate_latent(n_steps=10):
    """Interpolate between two random noise vectors to explore the latent space."""
    G.eval()
    
    z1 = torch.randn(1, LATENT_DIM, device=DEVICE)  # starting point
    z2 = torch.randn(1, LATENT_DIM, device=DEVICE)  # ending point
    
    # Linear interpolation: z = (1-t)*z1 + t*z2  for t ∈ [0, 1]
    alphas = torch.linspace(0, 1, n_steps, device=DEVICE)
    interpolated = [(1 - a) * z1 + a * z2 for a in alphas]
    interpolated = torch.cat(interpolated, dim=0)  # (n_steps, latent_dim)
    
    images = G(interpolated).view(n_steps, 1, 28, 28)
    grid = make_grid(images, nrow=n_steps, normalize=True, value_range=(-1, 1))
    
    plt.figure(figsize=(16, 2))
    plt.imshow(grid.cpu().permute(1, 2, 0), cmap='gray')
    plt.title('Latent space interpolation: z₁ → z₂ (smooth transition shows G has generalised)')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    G.train()

interpolate_latent(n_steps=12)

## Step 11 — Check for Mode Collapse

**Mode collapse** is one of the main failure modes of GANs: G learns to produce only one (or a few) digit(s) that reliably fool D, rather than learning the full distribution.

We generate a large batch and check the diversity across samples.

In [ ]:
@torch.no_grad()
def check_diversity(n_samples=256):
    """Generate many samples and visualize to check for mode collapse."""
    G.eval()
    z = torch.randn(n_samples, LATENT_DIM, device=DEVICE)
    images = G(z).view(n_samples, 1, 28, 28)
    
    # Pick 64 random samples for display
    idx = torch.randperm(n_samples)[:64]
    grid = make_grid(images[idx], nrow=8, normalize=True, value_range=(-1, 1))
    
    plt.figure(figsize=(12, 12))
    plt.imshow(grid.cpu().permute(1, 2, 0), cmap='gray')
    plt.title(f'Diversity check: 64 of {n_samples} generated samples\n'
              f'(mode collapse = all images look the same)')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Measure diversity: std of pixel values across samples
    imgs_flat = images.view(n_samples, -1).cpu().numpy()
    diversity = np.std(imgs_flat, axis=0).mean()
    print(f'Mean pixel std across {n_samples} samples: {diversity:.4f}')
    print(f'(higher = more diverse; ~0 = mode collapse)')
    G.train()

check_diversity()

## Step 12 — Save the Trained Models

Save G and D weights so you can load them later without retraining.

In [ ]:
torch.save({
    'epoch': NUM_EPOCHS,
    'generator_state_dict':     G.state_dict(),
    'discriminator_state_dict': D.state_dict(),
    'optimizer_G_state_dict':   optimizer_G.state_dict(),
    'optimizer_D_state_dict':   optimizer_D.state_dict(),
    'g_losses': g_losses,
    'd_losses': d_losses,
    'config': {
        'latent_dim': LATENT_DIM,
        'hidden_dim': HIDDEN_DIM,
        'img_dim': IMG_DIM,
    }
}, 'gan_mnist_checkpoint.pth')

print('Checkpoint saved to gan_mnist_checkpoint.pth')

# ── How to reload ────────────────────────────────────────────────────────────
# checkpoint = torch.load('gan_mnist_checkpoint.pth')
# G.load_state_dict(checkpoint['generator_state_dict'])
# D.load_state_dict(checkpoint['discriminator_state_dict'])

## Summary

| Component | What it does | Key detail |
|---|---|---|
| **Generator G** | z (100) → image (784) | Tanh output, LeakyReLU + BatchNorm hidden |
| **Discriminator D** | image (784) → scalar | Sigmoid output, LeakyReLU + Dropout hidden |
| **D loss** | `BCE(D(real),1) + BCE(D(fake),0)` | Maximizes V(D,G) |
| **G loss** | `BCE(D(G(z)),1)` | Non-saturating trick from paper |
| **Training** | Alternate D then G each batch | k=1 (as recommended in paper) |
| **Equilibrium** | D(x) ≈ 0.5 everywhere | p_g = p_data, JSD = 0 |

### What to try next

1. **DCGAN** — replace Linear layers with Conv/ConvTranspose for better image quality
2. **Conditional GAN** — add digit label as input to both G and D to control which digit is generated
3. **Wasserstein GAN (WGAN)** — replace BCE with Wasserstein distance for more stable training
4. **Different datasets** — CelebA, CIFAR-10, Fashion-MNIST